In [2]:
# Milvus vector database imports
import pymilvus
from pymilvus import MilvusClient  # High-level client for simple operations
from pymilvus import (
    utility,                        # Utility functions for Milvus
    FieldSchema,                    # Define schema fields
    CollectionSchema,               # Define collection structure
    DataType,                       # Data types for fields
    Collection,                     # Low-level collection operations
    AnnSearchRequest,               # Create search requests for hybrid search
    RRFRanker,                      # Reciprocal Rank Fusion reranker
    connections,                    # Manage Milvus connections
)

# Sparse embedding import 
from pymilvus.model.sparse.bm25.tokenizers import build_default_analyzer
from pymilvus.model.sparse import BM25EmbeddingFunction , SpladeEmbeddingFunction

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from datasets import load_dataset

sns.set_style("whitegrid")
plt.rcParams['figure.figsize']=(12,6)


/Users/nilasark/advanced/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
dataset = load_dataset("tdiggelm/climate_fever")
corpus_text=dataset['test']['claim']
for i,claim in enumerate(corpus_text[:5],1):
    print(f"{i}:{claim}")
print(f"\n Loaded total {len(corpus_text)} claims from the dataset")

# ============================================================
# Build and Fit BM25 Embedding Function
# ============================================================

# Build a default English language analyzer (tokenizer + stemmer)
# This will break text into tokens and normalize them (e.g., "running" → "run")
analyzer=build_default_analyzer(language='en')
bm25_ef=BM25EmbeddingFunction(analyzer)


# Create BM25 embedding function with the analyzer
# BM25 uses statistical Term Frequency-Inverse Document Frequency (TF-IDF)
print(f"fitting BM25 on conpus")
bm25_ef.fit(corpus_text)
print(f"\nBM25 fitted Vocabolary size:{bm25_ef.dim}" )


1:Global warming is driving polar bears toward extinction
2:The sun has gone into ‘lockdown’ which could cause freezing weather, earthquakes and famine, say scientists
3:The polar bear population has been growing.
4:Ironic' study finds more CO2 has slightly cooled the planet
5:Human additions of CO2 are in the margin of error of current measurements and the gradual increase in CO2 is mainly from oceans degassing as the planet slowly emerges from the last ice age.

 Loaded total 1535 claims from the dataset
fitting BM25 on conpus

BM25 fitted Vocabolary size:3410


In [4]:
documents = [
    "Currently, scientists none of these places, which today supply much of the world's food, will be reliable sources of any.",
    "Over the coming 25 or 30 years, scientists say, the climate is likely to gradually warm. However, researchers also say that this phenomenon can be stopped if human emissions are reduced to zero.",
    "The jet stream forms a boundary between the cold north and the warmer south, but the lower temperature difference means the winds are now weaker.",
    "Coral becomes stressed and expels the algae, which leave the coral a bleached white color.",
    "The rapid changes in the climate may have profound consequences for humans and other species... Severe drought caused food shortages for millions of people in Ethiopia, with a lack of rainfall resulting in intense and widespread forest fires in Indonesia that belched out a vast quantity of greenhouse gas"
]

print(f"Prepared {len(documents)} from embedding\n")

print(f"Creating BM25 embeddings")
bm25_embeddings_docs=bm25_ef.encode_documents(documents)

print(f"Sparse Vector Dimension: {bm25_ef.dim}")
print(f"{list(bm25_embeddings_docs)[0].shape}")

print(f"\nCreating Splade Embeddings")
splade_ef=SpladeEmbeddingFunction(
    model_name="naver/splade-v3",
    device="cpu"
)
splade_embedding_docs=splade_ef.encode_documents(documents)
print(f"SPLADE Vector Dimension: {splade_ef.dim}")
print(f"Splade Vector Shape:{list(splade_embedding_docs)[0].shape}")

Prepared 5 from embedding

Creating BM25 embeddings
Sparse Vector Dimension: 3410
(3410,)

Creating Splade Embeddings
SPLADE Vector Dimension: 30522
Splade Vector Shape:(30522,)


In [ ]:
print(f"Connecting to Milvus")

connections.connect(
     alias="default", 
    host="localhost",
    port="19530"
)

print("Connected To milvus")
fields=[
    FieldSchema(
        name="pk",
        dtype=DataType.VARCHAR,
        is_primary=True,
        auto_id=True,
        max_length=100
    ),
    FieldSchema(
        name="text",
        dtype=DataType.VARCHAR,
        max_length=512,
    ),
    FieldSchema(
        name="BM25_Vector",
        dtype=DataType.SPARSE_FLOAT_VECTOR,
    ),

    FieldSchema(
        name="SPLADE_Vector",
        dtype=DataType.SPARSE_FLOAT_VECTOR
    )
]

schema=CollectionSchema(
    fields=fields,
    description="Used for SPARSE and BM25 Vector Storage"
)

collection=Collection("bm25_splade",schema)
print(f"Collection Created BM25_SPLADE\n")

print(f"Creating Indices ...")

sparse_index= {
    "index_type":"SPARSE_INVERTED_INDEX",
    "metric_type":"IP"
}
# index_type: "SPARSE_INVERTED_INDEX"
# This tells the database to build an inverted index optimized for sparse vectors.

# A sparse vector might look conceptually like:

# Vector A = [0, 0, 0.8, 0, 0, 0.3, 0, ...]
#                       ↑        ↑
#                   non-zero values

# Instead of storing/searching thousands of zero values, the inverted index focuses on the dimensions that actually have non-zero values.
# Conceptually:

# dimension 2 → [(doc1, 0.8), (doc7, 0.5)]
# dimension 5 → [(doc1, 0.3), (doc3, 0.9)]
# dimension 9 → [(doc2, 0.7)]

# This makes retrieval efficient when vectors have many dimensions but relatively few non-zero values, which is common with sparse embedding models such as SPLADE or traditional term-weight representations.

# metric_type: "IP" tells the vector database how to calculate the similarity score between your query sparse vector and each stored sparse vector.

# IP = Inner Product, also called the dot product.

# Suppose you have:

# Query vector:
# Q = [0, 0.8, 0, 0.3]

# Document vector:
# D = [0, 0.6, 0, 0.5]


# Q = [0, 0.8, 0, 0.3]
#          ↓         ↓
# D = [0, 0.6, 0, 0.5]

# score = (0.8 × 0.6) + (0.3 × 0.5)
#       = 0.48 + 0.15
#       = 0.63

# The database then uses this score to rank documents. Typically, a larger IP score means a better match.

bm25_index=collection.create_index("BM25_Vector",sparse_index)
print(f"BM25 Index created")
splade_index=collection.create_index("SPLADE_Vector",sparse_index)
print(f"Splade Index created")

def sparse_to_dict(sparse_matrix):
    if hasattr(sparse_matrix,'tocoo'):
        coo=sparse_matrix.tocoo()

        return {
        int(idx):float(score) for idx,score in zip(coo.col,coo.data)
        }
    return sparse_matrix

print(f"Converting the Vector to Milvus Format")
bm25_embeddings_dict=[sparse_to_dict(vec) for vec in bm25_embeddings_docs]
splade_embeddings_dict=[sparse_to_dict(vec) for vec in splade_embedding_docs]

entities=[
    documents,
    bm25_embeddings_dict,
    splade_embeddings_dict
]

insert_result=collection.insert(entities)
collection.flush()


print(f"Collection now contains {collection.num_entities} entities")
print('\n')
print('*'*80)
print("Database Setup Complete")
print("*"*80)

Connecting to Milvus
Connected To milvus
Collection Created BM25_SPLADE

Creating Indices ...


/var/folders/j_/jb1h_wtd21zfymdrgw0tg8500000gn/T/ipykernel_59106/1234920705.py:3: PyMilvusDeprecationWarning: `connections.connect` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  connections.connect(
/var/folders/j_/jb1h_wtd21zfymdrgw0tg8500000gn/T/ipykernel_59106/1234920705.py:39: PyMilvusDeprecationWarning: `Collection` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  collection=Collection("bm25_splade",schema)
/var/folders/j_/jb1h_wtd21zfymdrgw0tg8500000gn/T/ipykernel_59106/1234920705.py:89: PyMilvusDeprecationWarning: `Collection.create_index` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  bm25_index=collection.create_index("BM25_Vector",sparse_index)


BM25 Index created


/var/folders/j_/jb1h_wtd21zfymdrgw0tg8500000gn/T/ipykernel_59106/1234920705.py:91: PyMilvusDeprecationWarning: `Collection.create_index` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  splade_index=collection.create_index("SPLADE_Vector",sparse_index)


Splade Index created
Converting the Vector to Milvus Format


/var/folders/j_/jb1h_wtd21zfymdrgw0tg8500000gn/T/ipykernel_59106/1234920705.py:113: PyMilvusDeprecationWarning: `Collection.insert` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  insert_result=collection.insert(entities)
/var/folders/j_/jb1h_wtd21zfymdrgw0tg8500000gn/T/ipykernel_59106/1234920705.py:114: PyMilvusDeprecationWarning: `Collection.flush` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  collection.flush()


Collection now coontains 5 entities


********************************************************************************
Database Setup Complete
********************************************************************************


/var/folders/j_/jb1h_wtd21zfymdrgw0tg8500000gn/T/ipykernel_59106/1234920705.py:117: PyMilvusDeprecationWarning: `Collection.num_entities` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  print(f"Collection now coontains {collection.num_entities} entities")


In [11]:
client=MilvusClient(uri="http://localhost:19530")
collection=Collection("bm25_splade")
collection.load()

print(f"Collection Loaded")

def perform_search(query:str,method:str="bm25",limit:int=3):
    splade_data=sparse_to_dict((splade_ef.encode_queries([query]))[0])
    bm25_data=sparse_to_dict((bm25_ef.encode_queries([query]))[0])
    if method=="bm25":
        
        result=client.search(
            collection_name="bm25_splade",
            data=[bm25_data],
            anns_field="BM25_Vector",
            limit=limit,
            output_fields=["text"]
        )
        return result

    elif method=="splade":
        result=client.search(
            collection_name="bm25_splade",
            data=[splade_data],
            anns_field="SPLADE_Vector",
            limit=limit,
            output_fields=["text"]
        )
        return result

    elif method=="hybrid":
        bm25_results=AnnSearchRequest(
            data=[bm25_data],
            anns_field="BM25_Vector",
            limit=limit,
            param={"metric_type":"IP"}
        )
        splade_results=AnnSearchRequest(
                    data=[splade_data],
                    anns_field="SPLADE_Vector",
                    limit=limit,
                    param={"metric_type":"IP"}
                )
        results=collection.hybrid_search(
           reqs=[bm25_results,splade_results],
           rerank=RRFRanker(k=60),
           limit=limit,
           output_fields=["text"]
        )
        return results
        


Collection Loaded


/var/folders/j_/jb1h_wtd21zfymdrgw0tg8500000gn/T/ipykernel_59106/1418418070.py:2: PyMilvusDeprecationWarning: `Collection` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  collection=Collection("bm25_splade")
/var/folders/j_/jb1h_wtd21zfymdrgw0tg8500000gn/T/ipykernel_59106/1418418070.py:3: PyMilvusDeprecationWarning: `Collection.load` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  collection.load()


In [12]:
def display_results(results,method_name):
    print(f"\n{'='*80}")
    print(f"  {method_name} RESULTS")
    print(f"{'='*80}")

    for i,content in enumerate(results[0],1):
        score=content['distance']
        text=content['entity']['text'] if 'entity' in content else content.get('text','')

        print(f"\n[Rank {i}] Score: {score:.4f}")
        print(f"  {text}")
    
    print(f"\n{'='*80}\n")

def compare_all_result(query,limit=3):
    print(f"\n{'='*80}")
    print(f"  {query} QUERY")
    print(f"{'='*80}")

    bm25_results=perform_search(query,"bm25",limit=3)
    splade_results=perform_search(query,"splade",limit=3)
    hybrid_results=perform_search(query,"hybrid",limit=3)

    display_results(bm25_results,"BM25")
    display_results(splade_results,"SPLADE")
    display_results(hybrid_results,"HYBRID")

    return bm25_results,splade_results,hybrid_results




In [13]:
query1 = "What do researchers say about global warming?"
q1_bm25, q1_splade, q1_hybrid = compare_all_result(query1, limit=3)


  What do researchers say about global warming? QUERY

  BM25 RESULTS

[Rank 1] Score: 8.4381
  Over the coming 25 or 30 years, scientists say, the climate is likely to gradually warm. However, researchers also say that this phenomenon can be stopped if human emissions are reduced to zero.



  SPLADE RESULTS

[Rank 1] Score: 15.9787
  Over the coming 25 or 30 years, scientists say, the climate is likely to gradually warm. However, researchers also say that this phenomenon can be stopped if human emissions are reduced to zero.

[Rank 2] Score: 10.1804
  The rapid changes in the climate may have profound consequences for humans and other species... Severe drought caused food shortages for millions of people in Ethiopia, with a lack of rainfall resulting in intense and widespread forest fires in Indonesia that belched out a vast quantity of greenhouse gas

[Rank 3] Score: 6.8139
  Currently, scientists none of these places, which today supply much of the world's food, will be reliable s

/var/folders/j_/jb1h_wtd21zfymdrgw0tg8500000gn/T/ipykernel_59106/1418418070.py:44: PyMilvusDeprecationWarning: `Collection.hybrid_search` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  results=collection.hybrid_search(
